In [ ]:
import pandas as pd
import pyarrow.parquet as pq

path = "../data/food.parquet"
parquet_file = pq.ParquetFile(path)

number_rows = parquet_file.metadata.num_rows
number_cols = parquet_file.metadata.num_columns

print(f"Le fichier se compose de {number_rows:,} lignes et de {number_cols} colonnes.")

schema_dataframe = pd.DataFrame({
    "column": parquet_file.schema_arrow.names,
    "type": [str(t) for t in parquet_file.schema_arrow.types]
})

# Display column data types
schema_dataframe

Le fichier se compose de 4,636,471 lignes et de 145 colonnes.


,column,type
0,additives_n,int32
1,additives_tags,list<element: string>
2,allergens_tags,list<element: string>
3,brands_tags,list<element: string>
4,brands,string
...,...,...
106,unknown_nutrients_tags,list<element: string>
107,vitamins_tags,list<element: string>
108,with_non_nutritive_sweeteners,int32
109,with_sweeteners,int32


### Affichage des ressources consommées en mémoire

In [12]:
# Load one row group as a real pandas DataFrame to measure actual memory usage
first_batch = next(parquet_file.iter_batches(batch_size=200_000))
datafram_sample = first_batch.to_pandas()

# memory_usage(deep=True) accounts for the real size of object dtypes (strings, lists),
# not just a fixed size per column — critical here since most OFF columns are text/tags
memory_sample_mb = datafram_sample.memory_usage(deep=True).sum() / (1024**2)
memory_per_row = datafram_sample.memory_usage(deep=True).sum() / len(datafram_sample)

# Extrapolate from the sample to the full file to estimate RAM needed for a full load
memory_estimate_gb = (memory_per_row * number_rows) / (1024**3)

# Note: this estimate will be well above the 7GB on-disk file size, since Parquet is
# compressed and pandas stores strings far less efficiently — this is the justification
# for chunked loading instead of pd.read_parquet() on the full file
print(f"Échantillon de {len(datafram_sample):,} lignes : {memory_sample_mb:.1f} Mo en mémoire")
print(f"Mémoire estimée pour les {number_rows:,} lignes du fichier complet : {memory_estimate_gb:.1f} Go")

Échantillon de 200,000 lignes : 2509.5 Mo en mémoire
Mémoire estimée pour les 4,636,471 lignes du fichier complet : 56.8 Go


### Affichage du taux de remplissage par colonne

In [13]:
# Compute fill rate (non-null ratio) for each column, processed in chunks
null_counts = pd.Series(0, index=parquet_file.schema_arrow.names)
rows_processed = 0

for batch in parquet_file.iter_batches(batch_size=200_000):
    dataframe_chunk = batch.to_pandas()
    rows_processed += len(dataframe_chunk)
    null_counts += dataframe_chunk.isna().sum()

fill_rate = (1 - null_counts / rows_processed).sort_values(ascending=False)
fill_rate_df = fill_rate.rename("fill_rate").to_frame()
fill_rate_df["fill_rate_pct"] = (fill_rate_df["fill_rate"] * 100).round(1)

fill_rate_df

,fill_rate,fill_rate_pct
code,1.000000,100.0
ingredients_text,1.000000,100.0
images,1.000000,100.0
generic_name,1.000000,100.0
product_name,1.000000,100.0
...,...,...
editors,0.012442,1.2
with_non_nutritive_sweeteners,0.006844,0.7
with_sweeteners,0.001048,0.1
photographers,0.000958,0.1
